<a href="https://colab.research.google.com/github/TuantdUIT/feedback_loop_v2/blob/main/models/training/phobert_ner_address.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune `kiendt/phobert-ner-address` sang nhãn L1–L7 (BIO + truncated)

Model gốc được train với 21 nhãn riêng (`LABEL_0`…`LABEL_20`), không khớp schema của prompt v2. Notebook này:

1. giữ lại **encoder** PhoBERT đã học địa chỉ của `kiendt/phobert-ner-address`, bỏ head 21 nhãn cũ;
2. gắn **2 head mới**: BIO 15 nhãn (`O`, `B/I-L1`…`B/I-L7`) và `truncated` nhị phân per-token;
3. train trên `models/data/phobert_ner/train.jsonl` (80k mẫu, sinh bởi `scripts/build_phobert_ner_dataset.py`), chọn checkpoint trên tập dev tách từ train theo `source_id`;
4. đo trên `test.jsonl` (20k, `source_id` + tên đường không có trong train) và trên **golden dataset** bằng đúng thước của `scripts/eval_golden.py`;
5. xuất hàm `parse(text)` trả JSON đúng schema prompt §6.2: `input`, `tokens`, `bio`, `spans[level,start,end,text,truncated]`.

**File cần có** — trên Colab đẩy vào `/content` (xem mục 1):
- `train.jsonl`, `test.jsonl` từ `models/data/phobert_ner/` trong repo; chạy local thì notebook tự đọc ở đó.
- `golden_*.json` (không bắt buộc): để đo trên golden.

Đơn vị token là **âm tiết** tách theo cùng regex với generator (`[^\s,\.;:()]+|[,\.;:()]`) — không dùng VnCoreNLP word-segment, để offset ký tự khớp tuyệt đối với input gốc.

In [1]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 141.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.2 MB/s eta 0:00:00


## 1. Lấy dữ liệu từ `/content` và cấu hình
Đẩy file vào `/content` (thư mục làm việc mặc định của Colab), bằng một trong hai cách:
- kéo thả vào panel **Files** bên trái (nhanh hơn với `train.jsonl` ~47MB), hoặc
- chạy cell dưới: nếu chưa thấy file, nó mở hộp chọn file để upload.

Cần: `train.jsonl`, `test.jsonl` (lấy từ `models/data/phobert_ner/` trong repo). Tuỳ chọn: `golden_full.json`, `golden_uncomplete.json`, `golden_test.json`, `golden_test2.json` để đo trên golden. Cũng nhận file `.zip` chứa các file trên — cell tự giải nén.

In [2]:
import shutil
import zipfile
from pathlib import Path

CONTENT = Path("/content")
REQUIRED = ["train.jsonl", "test.jsonl"]
GOLDEN_FILES = ["golden_full.json", "golden_uncomplete.json", "golden_test.json", "golden_test2.json"]


def unzip_all(folder):
    for z in folder.glob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
        print(f"giải nén {z.name}")
    # file nằm trong thư mục con sau khi giải nén (vd phobert_ner/train.jsonl) -> đưa ra /content
    for name in REQUIRED + GOLDEN_FILES:
        if not (folder / name).exists():
            found = next(folder.rglob(name), None)
            if found is not None and "drive" not in found.parts:
                shutil.copy2(found, folder / name)


if CONTENT.exists():
    unzip_all(CONTENT)
    missing = [f for f in REQUIRED if not (CONTENT / f).exists()]
    if missing:
        from google.colab import files
        print(f"Chưa có {missing} trong /content — chọn file để upload "
              f"(có thể chọn thêm {', '.join(GOLDEN_FILES)} hoặc một file .zip)")
        for name, data in files.upload().items():
            (CONTENT / name).write_bytes(data)
        unzip_all(CONTENT)
    missing = [f for f in REQUIRED if not (CONTENT / f).exists()]
    assert not missing, f"Vẫn thiếu {missing} trong /content"
    for f in REQUIRED + GOLDEN_FILES:
        if (CONTENT / f).exists():
            print(f"  /content/{f:24s} {(CONTENT / f).stat().st_size / 1e6:7.1f} MB")
else:
    print("Không chạy trên Colab — cell cấu hình sẽ tự dò dữ liệu local (models/data/phobert_ner).")

  /content/train.jsonl                 47.3 MB
  /content/test.jsonl                  12.2 MB


In [3]:
import json
import os
import random
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

MODEL_ID = "kiendt/phobert-ner-address"
SEED = 42
MAX_LEN = 128            # số subword tối đa / câu (PhoBERT cho phép 256); địa chỉ dài nhất ~60 subword
BATCH_SIZE = 64
EPOCHS = 3
LR_ENCODER = 3e-5
LR_HEADS = 5e-4          # head mới khởi tạo ngẫu nhiên -> lr cao hơn encoder
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
TRUNC_LOSS_WEIGHT = 0.5
TRUNC_THRESHOLD = 0.5
DEV_RATIO = 0.05         # tách từ train theo source_id để chọn checkpoint; test chỉ đo một lần ở cuối
MAX_TRAIN_ROWS = None    # đặt số nhỏ (vd 3000) để chạy thử nhanh
LOG_EVERY = 200


def find_dir(cands, marker, required=True):
    # thư mục đầu tiên CÓ file `marker` (không chỉ tồn tại thư mục — /content trên Colab luôn tồn tại)
    for c in cands:
        if (c / marker).exists():
            return c.resolve()
    if required:
        raise FileNotFoundError(f"Không thấy {marker} trong: " + ", ".join(map(str, cands)))
    return None


DATA_DIR = find_dir([
    Path("/content"),                             # Colab: file đẩy thẳng vào /content (cell trên)
    Path("/content/phobert_ner"),
    Path("/content/drive/MyDrive/phobert_ner"),   # Colab + Google Drive
    Path("/kaggle/input/phobert-ner"),            # Kaggle dataset
    Path("../data/phobert_ner"),                  # local: notebook nằm ở models/training/
    Path("models/data/phobert_ner"),              # local: chạy từ gốc repo
], "train.jsonl")
GOLDEN_DIR = find_dir([
    Path("/content"), Path("/content/golden_dataset"), Path("/content/drive/MyDrive/golden_dataset"),
    Path("/kaggle/input/golden-dataset"), Path("../../golden_dataset"), Path("golden_dataset"),
], "golden_full.json", required=False)
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/phobert_ner_l1l7")
elif Path("/content/drive/MyDrive").exists():
    OUTPUT_DIR = Path("/content/drive/MyDrive/phobert_ner_l1l7")
elif Path("/content").exists():
    OUTPUT_DIR = Path("/content/phobert_ner_l1l7")
else:
    OUTPUT_DIR = (DATA_DIR.parent.parent / "checkpoints" / "phobert_ner_l1l7").resolve()   # models/checkpoints/

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"DATA_DIR   = {DATA_DIR}\nGOLDEN_DIR = {GOLDEN_DIR}\nOUTPUT_DIR = {OUTPUT_DIR}\nDEVICE     = {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if USE_AMP else ""))

DATA_DIR   = /content
GOLDEN_DIR = None
OUTPUT_DIR = /content/phobert_ner_l1l7
DEVICE     = cuda (NVIDIA A100-SXM4-80GB)


## 2. Nạp dữ liệu, tách dev theo `source_id`
Mọi tiền tố của cùng một địa chỉ gốc nằm trọn trong train hoặc dev — nếu tách theo dòng, dev sẽ chứa gần như nguyên văn câu đã thấy khi train và F1 dev bị thổi phồng.

In [4]:
def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_all = read_jsonl(DATA_DIR / "train.jsonl")
test_rows = read_jsonl(DATA_DIR / "test.jsonl")

ids = sorted({r["source_id"] for r in train_all})
random.Random(SEED).shuffle(ids)
dev_ids = set(ids[: int(len(ids) * DEV_RATIO)])
dev_rows = [r for r in train_all if r["source_id"] in dev_ids]
train_rows = [r for r in train_all if r["source_id"] not in dev_ids]
if MAX_TRAIN_ROWS:
    train_rows = train_rows[:MAX_TRAIN_ROWS]
    dev_rows = dev_rows[: max(200, MAX_TRAIN_ROWS // 10)]

for name, rows in (("train", train_rows), ("dev", dev_rows), ("test", test_rows)):
    full = sum(r["is_full"] for r in rows)
    trunc = sum(any(s["truncated"] for s in r["spans"]) for r in rows)
    print(f"{name:5s} {len(rows):6d} mẫu | đầy đủ {full:6d} | tiền tố {len(rows) - full:6d} | có span truncated {trunc:6d}")
print()
print(json.dumps({k: train_rows[0][k] for k in ("text", "tokens", "bio", "trunc", "spans")}, ensure_ascii=False))

train  75964 mẫu | đầy đủ  18680 | tiền tố  57284 | có span truncated  28628
dev     4036 mẫu | đầy đủ   1028 | tiền tố   3008 | có span truncated   1537
test   20000 mẫu | đầy đủ   5298 | tiền tố  14702 | có span truncated   7384

{"text": "395/70/37 đường số 385 p. hà", "tokens": ["395/70/37", "đường", "số", "385", "p", ".", "hà"], "bio": ["B-L6", "B-L5", "I-L5", "I-L5", "B-L4", "I-L4", "I-L4"], "trunc": [0, 0, 0, 0, 0, 0, 1], "spans": [{"level": "L6", "start": 0, "end": 9, "text": "395/70/37", "truncated": false}, {"level": "L5", "start": 10, "end": 22, "text": "đường số 385", "truncated": false}, {"level": "L4", "start": 23, "end": 28, "text": "p. hà", "truncated": true}]}


## 3. Nhãn và tokenize
Mỗi âm tiết (token của dataset) được PhoBERT BPE tách thành 1+ subword. Nhãn BIO và `trunc` đặt ở **subword đầu** của mỗi âm tiết, các subword còn lại `-100` (không tính loss). Tokenizer của PhoBERT là bản slow (không có `word_ids()`), nên căn chỉnh làm tay và cache theo từ.

In [5]:
LEVELS = [f"L{i}" for i in range(1, 8)]
LABELS = ["O"] + [f"{p}-{lv}" for lv in LEVELS for p in ("B", "I")]
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

TOKEN_RE = re.compile(r"[^\s,\.;:()]+|[,\.;:()]")    # giống scripts/build_phobert_ner_dataset.py


def nfc(s):
    return unicodedata.normalize("NFC", s)


def word_tokenize(text):
    # text phải đã NFC -> (token, start, end) với offset ký tự trong text
    return [(m.group(0), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
_piece_cache = {}


def word_pieces(word):
    ids = _piece_cache.get(word)
    if ids is None:
        ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(word)) or [tokenizer.unk_token_id]
        _piece_cache[word] = ids
    return ids


def encode(words, bio=None, trunc=None):
    input_ids, word_index = [tokenizer.cls_token_id], [-1]
    labels, trunc_labels = [-100], [-100.0]
    for i, w in enumerate(words):
        pieces = word_pieces(w)
        if len(input_ids) + len(pieces) > MAX_LEN - 1:
            break                                     # phần vượt MAX_LEN -> dự đoán "O"
        for k, pid in enumerate(pieces):
            first = k == 0
            input_ids.append(pid)
            word_index.append(i if first else -1)
            labels.append(LABEL2ID[bio[i]] if bio is not None and first else -100)
            trunc_labels.append(float(trunc[i]) if trunc is not None and first else -100.0)
    input_ids.append(tokenizer.sep_token_id)
    word_index.append(-1)
    labels.append(-100)
    trunc_labels.append(-100.0)
    return {"input_ids": input_ids, "word_index": word_index, "labels": labels, "trunc_labels": trunc_labels}


def encode_rows(rows):
    return [encode(r["tokens"], r["bio"], r["trunc"]) for r in rows]


t0 = time.time()
train_enc, dev_enc = encode_rows(train_rows), encode_rows(dev_rows)
lens = np.array([len(e["input_ids"]) for e in train_enc])
n_words = sum(len(r["tokens"]) for r in train_rows)
n_cut = sum(sum(1 for w in e["word_index"] if w >= 0) < len(r["tokens"]) for e, r in zip(train_enc, train_rows))
unk = sum(ids == [tokenizer.unk_token_id] for ids in _piece_cache.values())
print(f"encode {time.time() - t0:.1f}s | subword/câu: mean {lens.mean():.1f}, p99 {np.percentile(lens, 99):.0f}, max {lens.max()}"
      f" | câu bị cắt do MAX_LEN: {n_cut} | từ -> <unk>: {unk}/{len(_piece_cache)} từ khác nhau")
print(tokenizer.tokenize("444 Nguyễn Trãi Quận Li"))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

encode 3.4s | subword/câu: mean 10.3, p99 26, max 48 | câu bị cắt do MAX_LEN: 0 | từ -> <unk>: 3/25248 từ khác nhau
['444', 'Nguyễn', 'Tr@@', 'ãi', 'Quận', 'Li']


In [6]:
def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, v: seq + [v] * (width - len(seq))          # noqa: E731
    out = {
        "input_ids": torch.tensor([pad(b["input_ids"], tokenizer.pad_token_id) for b in batch]),
        "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch]),
        "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
        "trunc_labels": torch.tensor([pad(b["trunc_labels"], -100.0) for b in batch]),
    }
    out["word_index"] = [b["word_index"] for b in batch]
    return out


train_loader = DataLoader(train_enc, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                          num_workers=0 if os.name == "nt" else 2)   # Windows: worker không pickle được hàm của notebook
print(f"{len(train_loader)} batch/epoch")

1187 batch/epoch


## 4. Model: encoder của `kiendt/phobert-ner-address` + 2 head mới
`AutoModel.from_pretrained` chỉ nạp phần `roberta.*` của checkpoint; head `classifier` 21 nhãn cũ bị bỏ (transformers sẽ in cảnh báo "weights not used" — đúng ý đồ).

In [7]:
class PhoBertAddressNER(nn.Module):
    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.bio_head = nn.Linear(hidden, num_labels)
        self.trunc_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask, labels=None, trunc_labels=None):
        hs = self.dropout(self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)
        bio_logits = self.bio_head(hs)
        trunc_logits = self.trunc_head(hs).squeeze(-1)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(bio_logits.reshape(-1, bio_logits.size(-1)).float(), labels.reshape(-1),
                                   ignore_index=-100)
            mask = trunc_labels >= 0
            if mask.any():
                loss = loss + TRUNC_LOSS_WEIGHT * F.binary_cross_entropy_with_logits(
                    trunc_logits[mask].float(), trunc_labels[mask].float())
        return loss, bio_logits, trunc_logits


def save_model(model, path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    model.encoder.save_pretrained(path / "encoder")
    tokenizer.save_pretrained(path / "encoder")
    torch.save({"bio_head": model.bio_head.state_dict(), "trunc_head": model.trunc_head.state_dict()},
               path / "heads.pt")
    (path / "config.json").write_text(json.dumps(
        {"base_model": MODEL_ID, "labels": LABELS, "max_len": MAX_LEN, "trunc_threshold": TRUNC_THRESHOLD,
         "token_regex": TOKEN_RE.pattern}, ensure_ascii=False, indent=2), encoding="utf-8")


def load_model(path):
    path = Path(path)
    model = PhoBertAddressNER(AutoModel.from_pretrained(path / "encoder", add_pooling_layer=False))
    heads = torch.load(path / "heads.pt", map_location="cpu")
    model.bio_head.load_state_dict(heads["bio_head"])
    model.trunc_head.load_state_dict(heads["trunc_head"])
    return model.to(DEVICE).eval()


model = PhoBertAddressNER(AutoModel.from_pretrained(MODEL_ID, add_pooling_layer=False)).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M tham số")

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: kiendt/phobert-ner-address
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


134.4M tham số


## 5. Giải mã và đo
- **BIO**: Viterbi có ràng buộc chuyển trạng thái (`I-X` chỉ sau `B-X`/`I-X`, không mở đầu bằng `I-X`) — đầu ra luôn hợp lệ theo prompt §3.6, không cần vá.
- **truncated**: chỉ xét span **cuối cùng** và chỉ khi span đó chạm mép phải chuỗi (prompt §3.5), lấy xác suất head `trunc` ở token cuối của span.
- Chỉ số trên tập tổng hợp: span khớp tuyệt đối `(level, start, end)`, `truncated` P/R/F1, và exact match cả bản ghi (spans + truncated).

In [8]:
_ALLOWED = np.zeros((NUM_LABELS, NUM_LABELS))
_START = np.zeros(NUM_LABELS)
for j, lab in enumerate(LABELS):
    if lab.startswith("I-"):
        _START[j] = -1e9
        for i, prev in enumerate(LABELS):
            if prev not in (f"B-{lab[2:]}", lab):
                _ALLOWED[i, j] = -1e9


def viterbi(logp):
    score = logp[0] + _START
    back = np.zeros(logp.shape, dtype=np.int64)
    for t in range(1, len(logp)):
        cand = score[:, None] + _ALLOWED
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(len(logp) - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return [LABELS[i] for i in reversed(path)]


def tags_to_spans(text, toks, tags, trunc_prob):
    spans = []
    for i, ((_, s, e), tag) in enumerate(zip(toks, tags)):
        if tag.startswith("B-"):
            spans.append({"level": tag[2:], "start": s, "end": e, "_last": i})
        elif tag.startswith("I-"):
            spans[-1]["end"], spans[-1]["_last"] = e, i
    edge = len(text.rstrip())
    for k, sp in enumerate(spans):
        last = sp.pop("_last")
        sp["text"] = text[sp["start"]:sp["end"]]
        sp["truncated"] = bool(k == len(spans) - 1 and sp["end"] == edge and trunc_prob[last] > TRUNC_THRESHOLD)
    return spans


@torch.inference_mode()
def predict(model, texts, batch_size=256):
    model.eval()
    out = []
    for b in range(0, len(texts), batch_size):
        chunk = [nfc(t) for t in texts[b:b + batch_size]]
        toks_list = [word_tokenize(t) for t in chunk]
        batch = collate([encode([w for w, _, _ in toks]) for toks in toks_list])
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            _, bio_logits, trunc_logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        logp = F.log_softmax(bio_logits.float(), -1).cpu().numpy()
        tprob = torch.sigmoid(trunc_logits.float()).cpu().numpy()
        for j, (text, toks) in enumerate(zip(chunk, toks_list)):
            word_logp = np.full((len(toks), NUM_LABELS), -1e9)
            word_logp[:, LABEL2ID["O"]] = 0.0                # từ bị cắt do MAX_LEN -> "O"
            word_tp = np.zeros(len(toks))
            for pos, w in enumerate(batch["word_index"][j]):
                if w >= 0:
                    word_logp[w], word_tp[w] = logp[j, pos], tprob[j, pos]
            tags = viterbi(word_logp) if toks else []
            out.append({"input": text, "tokens": [w for w, _, _ in toks], "bio": tags,
                        "spans": tags_to_spans(text, toks, tags, word_tp)})
    return out


def prf(tp, n_pred, n_gold):
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    return {"p": p, "r": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "n_gold": n_gold}


def evaluate_rows(model, rows, preds=None):
    preds = preds or predict(model, [r["text"] for r in rows])
    tok_ok = tok_n = exact = 0
    c = Counter()
    per_level = {lv: Counter() for lv in LEVELS}
    for r, p in zip(rows, preds):
        tok_ok += sum(a == b for a, b in zip(r["bio"], p["bio"]))
        tok_n += len(r["bio"])
        g = {(s["level"], s["start"], s["end"]) for s in r["spans"]}
        q = {(s["level"], s["start"], s["end"]) for s in p["spans"]}
        c["tp"] += len(g & q); c["pred"] += len(q); c["gold"] += len(g)
        for lv in LEVELS:
            gl, ql = {x for x in g if x[0] == lv}, {x for x in q if x[0] == lv}
            per_level[lv]["tp"] += len(gl & ql); per_level[lv]["pred"] += len(ql); per_level[lv]["gold"] += len(gl)
        gt = {(s["level"], s["start"], s["end"]) for s in r["spans"] if s["truncated"]}
        pt = {(s["level"], s["start"], s["end"]) for s in p["spans"] if s["truncated"]}
        c["t_tp"] += len(gt & pt); c["t_pred"] += len(pt); c["t_gold"] += len(gt)
        full = lambda spans: {(s["level"], s["start"], s["end"], s["truncated"]) for s in spans}   # noqa: E731
        exact += full(r["spans"]) == full(p["spans"])
    return {
        "n": len(rows), "token_acc": tok_ok / max(1, tok_n), "exact": exact / max(1, len(rows)),
        "span": prf(c["tp"], c["pred"], c["gold"]), "truncated": prf(c["t_tp"], c["t_pred"], c["t_gold"]),
        "per_level": {lv: prf(v["tp"], v["pred"], v["gold"]) for lv, v in per_level.items() if v["gold"] or v["pred"]},
    }


def show(m, title):
    print(f"{title}: n={m['n']} | token acc {m['token_acc']:.4f} | span P/R/F1 {m['span']['p']:.4f}/"
          f"{m['span']['r']:.4f}/{m['span']['f1']:.4f} | truncated F1 {m['truncated']['f1']:.4f} "
          f"(n={m['truncated']['n_gold']}) | exact {m['exact']:.4f}")

## 6. Train
Chọn checkpoint tốt nhất theo span F1 trên dev, lưu vào `OUTPUT_DIR/best`.

In [9]:
no_decay = ("bias", "LayerNorm.weight")
enc_named = list(model.encoder.named_parameters())
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in enc_named if not any(k in n for k in no_decay)], "lr": LR_ENCODER, "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in enc_named if any(k in n for k in no_decay)], "lr": LR_ENCODER, "weight_decay": 0.0},
    {"params": list(model.bio_head.parameters()) + list(model.trunc_head.parameters()), "lr": LR_HEADS, "weight_decay": 0.0},
])
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history, best_f1 = [], -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, running = time.time(), 0.0
    for step, batch in enumerate(train_loader, 1):
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "word_index"}
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss, _, _ = model(**inputs)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
        if step % LOG_EVERY == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {running / step:.4f} ({time.time() - t0:.0f}s)")
    m = evaluate_rows(model, dev_rows)
    history.append({"epoch": epoch, "train_loss": running / len(train_loader), "dev": m})
    show(m, f"epoch {epoch} | train loss {running / len(train_loader):.4f} | dev")
    if m["span"]["f1"] > best_f1:
        best_f1 = m["span"]["f1"]
        save_model(model, OUTPUT_DIR / "best")
        print(f"  -> lưu best (dev span F1 {best_f1:.4f}) vào {OUTPUT_DIR / 'best'}")

(OUTPUT_DIR / "history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")

  epoch 1 step 200/1187 loss 1.1508 (12s)
  epoch 1 step 400/1187 loss 0.6624 (23s)
  epoch 1 step 600/1187 loss 0.4804 (33s)
  epoch 1 step 800/1187 loss 0.3847 (45s)
  epoch 1 step 1000/1187 loss 0.3253 (56s)
epoch 1 | train loss 0.2860 | dev: n=4036 | token acc 0.9820 | span P/R/F1 0.9637/0.9647/0.9642 | truncated F1 0.8534 (n=1537) | exact 0.8875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> lưu best (dev span F1 0.9642) vào /content/phobert_ner_l1l7/best
  epoch 2 step 200/1187 loss 0.0600 (11s)
  epoch 2 step 400/1187 loss 0.0597 (21s)
  epoch 2 step 600/1187 loss 0.0583 (31s)
  epoch 2 step 800/1187 loss 0.0568 (42s)
  epoch 2 step 1000/1187 loss 0.0552 (52s)
epoch 2 | train loss 0.0542 | dev: n=4036 | token acc 0.9839 | span P/R/F1 0.9687/0.9696/0.9691 | truncated F1 0.8818 (n=1537) | exact 0.9113


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> lưu best (dev span F1 0.9691) vào /content/phobert_ner_l1l7/best
  epoch 3 step 200/1187 loss 0.0369 (11s)
  epoch 3 step 400/1187 loss 0.0376 (21s)
  epoch 3 step 600/1187 loss 0.0369 (32s)
  epoch 3 step 800/1187 loss 0.0363 (42s)
  epoch 3 step 1000/1187 loss 0.0364 (52s)
epoch 3 | train loss 0.0362 | dev: n=4036 | token acc 0.9861 | span P/R/F1 0.9722/0.9723/0.9723 | truncated F1 0.8910 (n=1537) | exact 0.9215


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> lưu best (dev span F1 0.9723) vào /content/phobert_ner_l1l7/best


4685

## 7. Đo trên test tổng hợp (20k)
Test dùng `source_id` và tên đường (street_pool = heldout) chưa từng xuất hiện ở train — đây là số đo khả năng tổng quát hoá, không phải nhớ mặt chữ.

In [10]:
best = load_model(OUTPUT_DIR / "best")
test_preds = predict(best, [r["text"] for r in test_rows])
m_test = evaluate_rows(best, test_rows, test_preds)
show(m_test, "TEST")
for name, pick in (("  câu đầy đủ", lambda r: r["is_full"]), ("  tiền tố gõ dở", lambda r: not r["is_full"])):
    idx = [i for i, r in enumerate(test_rows) if pick(r)]
    show(evaluate_rows(best, [test_rows[i] for i in idx], [test_preds[i] for i in idx]), name)
print("\nTheo level:")
for lv, v in m_test["per_level"].items():
    print(f"  {lv}: P {v['p']:.4f} R {v['r']:.4f} F1 {v['f1']:.4f} (n={v['n_gold']})")
(OUTPUT_DIR / "metrics_test.json").write_text(json.dumps(m_test, ensure_ascii=False, indent=2), encoding="utf-8")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

TEST: n=20000 | token acc 0.9876 | span P/R/F1 0.9747/0.9744/0.9746 | truncated F1 0.8747 (n=7384) | exact 0.9096
  câu đầy đủ: n=5298 | token acc 0.9945 | span P/R/F1 0.9894/0.9896/0.9895 | truncated F1 0.0000 (n=0) | exact 0.9609
  tiền tố gõ dở: n=14702 | token acc 0.9813 | span P/R/F1 0.9641/0.9635/0.9638 | truncated F1 0.8767 (n=7384) | exact 0.8912

Theo level:
  L1: P 0.9916 R 0.9958 F1 0.9937 (n=236)
  L2: P 0.9770 R 0.9709 F1 0.9740 (n=5392)
  L3: P 0.9611 R 0.9760 F1 0.9685 (n=5492)
  L4: P 0.9568 R 0.9524 F1 0.9546 (n=7307)
  L5: P 0.9719 R 0.9685 F1 0.9702 (n=14707)
  L6: P 0.9949 R 0.9960 F1 0.9955 (n=12124)
  L7: P 0.9726 R 0.9734 F1 0.9730 (n=7405)


1282

In [11]:
# Một vài mẫu sai để soi
fmt = lambda spans: " | ".join(f"{s['level']}:{s['text']}{'*' if s['truncated'] else ''}" for s in spans)   # noqa: E731
wrong = [(r, p) for r, p in zip(test_rows, test_preds)
         if {(s["level"], s["start"], s["end"], s["truncated"]) for s in r["spans"]}
         != {(s["level"], s["start"], s["end"], s["truncated"]) for s in p["spans"]}]
print(f"{len(wrong)} / {len(test_rows)} mẫu sai (* = truncated)\n")
for r, p in random.Random(0).sample(wrong, min(15, len(wrong))):
    print(f"{r['text']!r}\n   gold: {fmt(r['spans'])}\n   pred: {fmt(p['spans'])}")

1807 / 20000 mẫu sai (* = truncated)

'bệnh viện phụ sản thích quảng đức bảo lộc lâm đồng'
   gold: L7:bệnh viện phụ sản | L5:thích quảng đức | L3:bảo lộc | L2:lâm đồng
   pred: L7:bệnh viện phụ sản thích | L5:quảng đức | L3:bảo lộc | L2:lâm đồng
'KIỆT 160/149 ĐƯỜNG NK'
   gold: L6:KIỆT 160/149 | L5:ĐƯỜNG NK
   pred: L6:KIỆT 160/149 | L5:ĐƯỜNG NK*
'212/119, P'
   gold: L6:212/119 | L5:P*
   pred: L6:212/119 | L4:P*
'quán bun bo chú tu ngu hành sơn quan hải châu da nang'
   gold: L7:quán bun bo chú tu | L4:ngu hành sơn | L3:quan hải châu | L2:da nang
   pred: L7:quán bun bo chú tu | L5:ngu hành sơn | L3:quan hải châu | L2:da nang
'207c quang dũng, lâm'
   gold: L6:207c | L5:quang dũng | L2:lâm
   pred: L6:207c | L5:quang dũng | L4:lâm
'29 lô 18c đường số 7, kp.7, bình quới, hcm'
   gold: L6:29 lô 18c | L5:đường số 7, kp.7 | L4:bình quới | L2:hcm
   pred: L6:29 lô 18c | L5:đường số 7, kp.7 | L3:bình quới | L2:hcm
'PHỔ QUANG K'
   gold: L5:PHỔ QUANG | L7:K*
   pred: L4:PHỔ QUANG K*
'Trườn

## 8. Đo trên golden dataset
Cùng thước với `scripts/eval_golden.py`: span = cặp `(level, text)` sau khi chuẩn hoá (NFC, chữ thường, gộp khoảng trắng, bỏ dấu câu hai đầu); Accuracy = khớp trọn cả 7 level. Golden không có nhãn `truncated` nên chỉ đo level + text.

`golden_test.json` / `golden_test2.json` là tập con của `golden_full` + `golden_uncomplete`; địa chỉ nguồn của golden đã bị loại khỏi dữ liệu train khi sinh, nên đây là số đo sạch.

In [12]:
def norm_text(s):
    s = re.sub(r"\s+", " ", nfc(s).lower())
    return s.strip(" ,.;:-")


def to_counter(pairs):
    return Counter((lv, norm_text(tx)) for lv, tx in pairs if norm_text(tx))


def eval_golden(model, path):
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    preds = predict(model, raw["texts"])
    exact, c, per_level, rows = 0, Counter(), {lv: Counter() for lv in LEVELS}, []
    for text, res, p in zip(raw["texts"], raw["results"], preds):
        g = to_counter((lv, tx) for lv in LEVELS for tx in res["result"].get(lv, []))
        q = to_counter((s["level"], s["text"]) for s in p["spans"])
        hit = g & q
        exact += g == q
        c["tp"] += sum(hit.values()); c["pred"] += sum(q.values()); c["gold"] += sum(g.values())
        for name, cnt in (("tp", hit), ("pred", q), ("gold", g)):
            for (lv, _), n in cnt.items():
                per_level[lv][name] += n
        rows.append({"text": text, "gold": sorted(g.elements()), "pred": sorted(q.elements()), "exact": g == q})
    return {"n": len(raw["texts"]), "accuracy": exact / len(raw["texts"]), "micro": prf(c["tp"], c["pred"], c["gold"]),
            "per_level": {lv: prf(v["tp"], v["pred"], v["gold"]) for lv, v in per_level.items() if v["gold"] or v["pred"]}}, rows


golden_metrics, golden_rows = {}, {}
if GOLDEN_DIR is None:
    print("Không thấy golden_dataset — bỏ qua bước này.")
else:
    for name in ("golden_full.json", "golden_uncomplete.json", "golden_test.json", "golden_test2.json"):
        if not (GOLDEN_DIR / name).exists():
            continue
        m, rows = eval_golden(best, GOLDEN_DIR / name)
        golden_metrics[name], golden_rows[name] = m, rows
        print(f"{name:24s} n={m['n']:4d} | acc {m['accuracy']:.3f} | P {m['micro']['p']:.3f} "
              f"R {m['micro']['r']:.3f} F1 {m['micro']['f1']:.3f} | "
              + " ".join(f"{lv}:{v['f1']:.2f}" for lv, v in m["per_level"].items()))
    (OUTPUT_DIR / "metrics_golden.json").write_text(json.dumps(golden_metrics, ensure_ascii=False, indent=2), encoding="utf-8")

Không thấy golden_dataset — bỏ qua bước này.


In [13]:
# Mẫu golden sai
for name, rows in golden_rows.items():
    bad = [r for r in rows if not r["exact"]]
    print(f"== {name}: {len(bad)} mẫu sai")
    for r in bad[:8]:
        print(f"  {r['text']!r}\n     gold: {r['gold']}\n     pred: {r['pred']}")

## 9. Dùng model: `parse(text)` trả JSON đúng schema prompt §6.2

In [14]:
def parse(text):
    return predict(best, [text])[0]


for t in ["444 Nguyễn Trãi Quận Li", "858 Đường", "858 Đường Nguyễn Huệ", "Highland - Cách Mạng Tháng 8",
          "Đại học Nguyễn Tất Thành", "62 - 64 Hai Bà Trưng", "4 38 39 phố Đại Đồng", "125/",
          "75C Thảo Nguyên, HN", "Thành phố Thủ Đức, Thành phố Hồ Chí Minh"]:
    print(json.dumps(parse(t), ensure_ascii=False))

{"input": "444 Nguyễn Trãi Quận Li", "tokens": ["444", "Nguyễn", "Trãi", "Quận", "Li"], "bio": ["B-L6", "B-L5", "I-L5", "B-L3", "I-L3"], "spans": [{"level": "L6", "start": 0, "end": 3, "text": "444", "truncated": false}, {"level": "L5", "start": 4, "end": 15, "text": "Nguyễn Trãi", "truncated": false}, {"level": "L3", "start": 16, "end": 23, "text": "Quận Li", "truncated": true}]}
{"input": "858 Đường", "tokens": ["858", "Đường"], "bio": ["B-L6", "B-L5"], "spans": [{"level": "L6", "start": 0, "end": 3, "text": "858", "truncated": false}, {"level": "L5", "start": 4, "end": 9, "text": "Đường", "truncated": true}]}
{"input": "858 Đường Nguyễn Huệ", "tokens": ["858", "Đường", "Nguyễn", "Huệ"], "bio": ["B-L6", "B-L5", "I-L5", "I-L5"], "spans": [{"level": "L6", "start": 0, "end": 3, "text": "858", "truncated": false}, {"level": "L5", "start": 4, "end": 20, "text": "Đường Nguyễn Huệ", "truncated": false}]}
{"input": "Highland - Cách Mạng Tháng 8", "tokens": ["Highland", "-", "Cách", "Mạng", "

In [15]:
# Nén checkpoint để tải về / chép sang máy khác (encoder + tokenizer + heads.pt + config.json)
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR / "phobert_ner_l1l7_best"), "zip", OUTPUT_DIR / "best")
print(archive)

/content/phobert_ner_l1l7/phobert_ner_l1l7_best.zip
